# 03 - Integração com LLM (Gemini)

Este notebook demonstra a geração de explicações de
diagnóstico em linguagem natural (`src/llm`), usando o modelo treinado e os resultados do algoritmo genético.

Cobre os dois casos de uso do Tech Challenge:
1. Explicação individual de uma predição de risco de AVC, para um paciente específico.
2. Interpretação agregada dos resultados do AG (baseline vs. otimizado), em insights
   acionáveis para profissionais de saúde.

Cada resposta gerada passa por um checklist determinístico de qualidade
(`evaluate_explanation`), sem depender de uma segunda chamada de LLM.

**Pré-requisitos:**
- Rodar `notebooks/01_baseline.ipynb` primeiro (gera os `.joblib` em `results/`).
- Rodar `notebooks/02_genetic_algorithm.ipynb` (gera `results/ga_summary.json`).
- Ter uma `GOOGLE_API_KEY` válida em um arquivo `.env` na raiz do projeto
  (copie `.env.example` e preencha).

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/stroke-prediction')
    sys.path.insert(0, str(ROOT))
    print("Rodando no Google Colab")
else:
    ROOT = Path('..').resolve()
    print("Rodando Localmente")

from src.llm import evaluate_explanation, explain_prediction, summarize_experiment
from src.models import load_model, predict
from src.preprocessing import prepare_pipeline

ga_summary_path = ROOT / 'results' / 'ga_summary.json'

if not ga_summary_path.exists():
    raise FileNotFoundError(
        'ga_summary.json nao encontrado. Rode notebooks/02_genetic_algorithm.ipynb primeiro.'
    )

with open(ga_summary_path, 'r', encoding='utf-8') as f:
    ga_summary = json.load(f)

print(f"Experimentos carregados: {len(ga_summary['experiments'])}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rodando no Google Colab
Experimentos carregados: 3


## 1. Explicações individuais de diagnóstico

Usamos o modelo de Regressão Logística (maior recall entre os baselines da Fase 1)
para prever o risco de 3 pacientes do conjunto de teste: os 2 com maior probabilidade
estimada de AVC, e 1 com a menor, para contraste.

In [2]:
model = load_model('logistic_regression')
_, X_test, _, y_test = prepare_pipeline()

predictions, probabilities = predict(model, X_test)
prob_series = pd.Series(probabilities, index=X_test.index)

sample_idx = list(prob_series.sort_values(ascending=False).index[:2]) + \
    list(prob_series.sort_values(ascending=True).index[:1])

Model loaded: /content/drive/MyDrive/stroke-prediction/results/logistic_regression.joblib
Loaded dataset: 5110 rows, 12 columns
Train size (after SMOTE): 7776 (positive: 3888, negative: 3888)
Test size: 1022 (positive: 50)


In [3]:
results_individual = []

for idx in sample_idx:
    row_pos = X_test.index.get_loc(idx)
    patient_data = X_test.loc[idx].to_dict()
    prediction = int(predictions[row_pos])
    probability = float(probabilities[row_pos])

    explanation = explain_prediction(patient_data, prediction, probability)
    evaluation = evaluate_explanation(explanation, patient_data)

    results_individual.append({
        'patient_index': idx,
        'prediction': prediction,
        'probability': probability,
        'explanation': explanation,
        'quality_score': evaluation['score'],
        'quality_checks': evaluation['checks'],
    })

    print(f"--- Paciente {idx} (predicao={prediction}, prob={probability:.1%}) ---")
    print(explanation)
    print(f"Qualidade: {evaluation['score']:.2f} | {evaluation['checks']}\n")

--- Paciente 2453 (predicao=1, prob=96.9%) ---
Olá, Doutor(a). Apresento a análise do modelo de inteligência artificial para o risco de Acidente Vascular Cerebral (AVC) do seu paciente.

### **Resultado da Predição**
O modelo classificou o paciente como de **alto risco de AVC (Classe 1)**, com uma probabilidade estimada de **96,9%**.

---

### **Fatores de Impacto Identificados**
Com base exclusivamente nos dados clínicos e demográficos fornecidos, os fatores que mais provavelmente influenciaram essa predição de risco tão elevada são:

* **Idade Avançada (82 anos):** A idade é um dos fatores de risco não modificáveis mais robustos para o AVC.
* **Glicemia Elevada (214,42 mg/dL):** O nível médio de glicose está significativamente alto, sugerindo um quadro de diabetes ou descontrole glicêmico agudo/crônico importante.
* **Obesidade (IMC de 33,9):** O Índice de Massa Corporal classifica o paciente em obesidade (grau I), o que sabidamente sobrecarrega o sistema cardiovascular.
* **Históric

## 2. Interpretação agregada dos experimentos do AG

Para cada um dos 3 experimentos registrados em `results/ga_summary.json`,
comparamos o modelo baseline com o modelo otimizado
pelo algoritmo genético e pedimos um resumo executivo em linguagem natural.

In [4]:
results_aggregate = []

for exp in ga_summary['experiments']:
    baseline_metrics = ga_summary['baseline'][exp['model_type']]
    optimized_metrics = exp['optimized_metrics']
    best_params = exp['ga']['best_params']

    summary_text = summarize_experiment(baseline_metrics, optimized_metrics, best_params)

    source_data = {**baseline_metrics, **optimized_metrics, **best_params}
    evaluation = evaluate_explanation(summary_text, source_data)

    results_aggregate.append({
        'experiment': exp['name'],
        'model_type': exp['model_type'],
        'summary': summary_text,
        'quality_score': evaluation['score'],
        'quality_checks': evaluation['checks'],
    })

    print(f"--- {exp['name']} ({exp['model_type'].upper()}) ---")
    print(summary_text)
    print(f"Qualidade: {evaluation['score']:.2f} | {evaluation['checks']}\n")

--- EXP-01 (LR) ---
**Relatório Executivo: Avaliação da Otimização do Modelo de Previsão de Risco de AVC**

**Para:** Equipe Médica e de Gestão de Dados  
**Assunto:** Análise de resultados da otimização por algoritmos genéticos (EXP-01)  

---

### 1. Resumo dos Resultados (Otimização vs. Baseline)

Após a aplicação do algoritmo genético para otimização de hiperparâmetros no modelo de Regressão Logística, **não houve ganho de desempenho**. Pelo contrário, o modelo otimizado apresentou uma leve regressão em todas as métricas de avaliação, incluindo a métrica de maior importância clínica: o **recall**.

Abaixo, apresentamos a comparação direta das métricas:

| Métrica | Modelo Original (Baseline) | Modelo Otimizado (EXP-01) | Variação Absoluta |
| :--- | :---: | :---: | :---: |
| **Recall** (Sensibilidade) | **0.4400** | **0.4200** | **-0.0200** |
| **F1-Score** | 0.2056 | 0.1972 | -0.0084 |
| **Acurácia** | 0.8337 | 0.8327 | -0.0010 |
| **Precisão** | 0.1341 | 0.1288 | -0.0053 |

---



## 3. Resumo da avaliação de qualidade

Consolida o score do checklist determinístico (`src/llm/evaluation.py`) para todas
as respostas geradas nesta demonstração, individuais e agregadas.

In [5]:
quality_rows = (
    [
        {'tipo': 'individual', 'id': str(r['patient_index']), **r['quality_checks'], 'score': r['quality_score']}
        for r in results_individual
    ]
    + [
        {'tipo': 'agregado', 'id': r['experiment'], **r['quality_checks'], 'score': r['quality_score']}
        for r in results_aggregate
    ]
)

df_quality = pd.DataFrame(quality_rows)
df_quality

,tipo,id,mentions_recall,grounded_in_data,uses_risk_language,reasonable_length,score
0,individual,2453,True,False,True,True,0.75
1,individual,5,True,True,True,True,1.00
2,individual,2740,True,True,True,True,1.00
3,agregado,EXP-01,True,True,True,True,1.00
4,agregado,EXP-02,True,True,True,True,1.00
5,agregado,EXP-03,True,True,True,True,1.00


## 4. Notas de prompt engineering e avaliação

- **Grounding**: os prompts (`src/llm/prompts.py`) só incluem dados que já existem no
  dataset ou nos resultados do AG.
- **Instrução explícita de cautela clínica**: os prompts pedem explicitamente que o
  modelo não substitua o julgamento clínico e não invente exames/históricos.
- **Avaliação determinística vs. LLM-as-judge**: optamos por um checklist de regras
  (`evaluate_explanation`) em vez de usar uma segunda chamada de LLM como "juiz". Isso
  torna a avaliação 100% reprodutível e testável em CI, ao custo de ser mais rígida.
  Ela não captura nuances de qualidade que fogem das 4 regras verificadas
  (menção ao recall quando aplicável, grounding numérico, linguagem de risco e
  tamanho da resposta).
- **Limitação conhecida**: o checklist é heurístico. Uma resposta pode ter score alto
  e ainda assim conter uma explicação clinicamente pobre (ou vice-versa). Ele serve
  como um filtro automático de sanidade, não como substituto de revisão humana.